# TimeGuard — GPU Backend

Bu notebook Colab GPU runtime'ında FastAPI backend'i çalıştırır.

**Hazırlık:**
1. Runtime → Change runtime type → **T4 GPU** seç
2. Tüm hücreleri **sırayla** çalıştır
3. Hücre 7'den sonra ngrok URL'ini kopyala
4. Yerel Streamlit sidebar'ına yapıştır

**Önemli:** Her hücrenin yanında yeşil tik işareti olana kadar bekle. Bir hücrede hata olursa dur ve düzelt.

In [ ]:
# ── Hücre 1: Kütüphane Kurulumu ──
!nvidia-smi

!pip install -q fastapi uvicorn mlflow "sqlalchemy<2.1" psycopg2-binary shap pydantic-settings pyngrok
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip install -q cuml-cu12

print('\nKurulum tamamlandı!')

In [ ]:
# ── Hücre 2: PostgreSQL Kurulumu ──
!apt-get update -qq && apt-get install -y -qq postgresql > /dev/null 2>&1
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres WITH PASSWORD 'postgres';"
!sudo -u postgres psql -c "CREATE DATABASE anomaly_detection OWNER postgres;"

print('PostgreSQL hazır')

In [ ]:
# ── Hücre 3: Proje Kodunu Yükle ──
!git clone https://github.com/afdemir06/anomaly-deteciton.git anomaly-detection 2>/dev/null || true
%cd anomaly-detection
!ls -la

In [ ]:
# ── Hücre 4: .env Dosyasını Oluştur ──
%%writefile .env
POSTGRES_HOST=localhost
POSTGRES_PORT=5432
POSTGRES_USER=postgres
POSTGRES_PASSWORD=postgres
POSTGRES_DB=anomaly_detection
MLFLOW_TRACKING_URI=http://localhost:5000
MLFLOW_EXPERIMENT_NAME=anomaly-detection
IF_N_ESTIMATORS=100
IF_CONTAMINATION=0.05
IF_RANDOM_STATE=42
DBSCAN_EPS=0.5
DBSCAN_MIN_SAMPLES=5
DBSCAN_METRIC=euclidean
LSTM_SEQ_LENGTH=30
LSTM_HIDDEN_DIM=64
LSTM_NUM_EPOCHS=25
LSTM_LEARNING_RATE=0.001
LSTM_THRESHOLD_PERCENTILE=95.0
ENSEMBLE_MIN_VOTES=2
LABEL_COLUMN=Class
DBSCAN_SUBSAMPLE_SIZE=10000
SHAP_MAX_DISPLAY=20
SHAP_SAMPLE_SIZE=100
API_HOST=0.0.0.0
API_PORT=8000
MLFLOW_HOST=localhost
MLFLOW_PORT=5000

print('.env dosyası oluşturuldu')

In [ ]:
# ── Hücre 5: MLflow Başlat ──
import subprocess, time

mlflow_proc = subprocess.Popen(
    ["mlflow", "server",
     "--backend-store-uri", "sqlite:///mlflow.db",
     "--default-artifact-root", "./mlruns",
     "--host", "localhost",
     "--port", "5000"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(5)
print(f'MLflow başlatıldı (pid={mlflow_proc.pid})')

In [ ]:
# ── Hücre 6: ngrok ile Public URL Oluştur ──
from pyngrok import ngrok

# ngrok token: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "SENIN_NGROK_TOKEN_BURAYA"  # ← BUNU DEĞİŞTİR

assert NGROK_TOKEN != "SENIN_NGROK_TOKEN_BURAYA", "Lütfen ngrok token'ını girin!"

ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(8000)
print(f'\nngrok URL: {tunnel.public_url}')
print('Bu URL\'yi Streamlit sidebar\'ına yapıştırın')

In [ ]:
# ── Hücre 7: API Başlat ve Test Et ──
import os, subprocess, sys, time

# Port 8000'i temizle (önceki instance varsa)
os.system("fuser -k 8000/tcp 2>/dev/null")
time.sleep(2)

# API'yi başlat
os.chdir("/content/anomaly-detection")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "api:app",
     "--host", "0.0.0.0", "--port", "8000"]
)
time.sleep(10)

# Kontrol et
poll = proc.poll()
if poll is not None:
    print(f'API ÇÖKTÜ! Return code: {poll}')
else:
    import requests
    try:
        r = requests.get("http://localhost:8000/docs", timeout=5)
        print(f'API BAŞARILI! status={r.status_code}')
        print('Artık Streamlit sidebar\'ına ngrok URL\'ini yapıştırabilirsin.')
    except Exception as e:
        print(f'API başlatılamadı: {e}')